# 06   GraphRAG Query API (Neo4j)

**Pipeline position:** `01` -> `02` -> `03a` -> `03b` -> `04` -> `05` -> **06**.
**Purpose:** expose and exercise the **GraphRAG query API** built on top of the `neo4j_graph` retrieval path (notebook 05 §7). A single entry point:

```
rag = GraphRAG()
result = rag.query(question)           # result.answer / .sources / .chunks
                                      # result.entities / .relationships / .cypher
                                      # result.retrieval_scores / .debug()
```

Implementation (`src/graphrag_n4j/rag.py`):
1. **retrieval**   `Neo4jGraphRetriever.search` (vector seeds -> bounded Cypher traversal)   already verified in notebook 05;
2. **context**   `graphrag_n4j.context.build_context` renders 5 sections (Documents, Chunks, Entities, Relationships, Provenance) with provenance on every line;
3. **generation**   `neo4j_config.make_llm()` (Ollama `qwen3.8:27b`) answers the question from the context using the `SYSTEM_PROMPT` in `graphrag_n4j.context`.


---
## 0   Setup


In [1]:
import os, sys, json, time
from pathlib import Path

from rich.console import Console
from rich import print as rprint

def _repo_root():
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "src"))

import common as c
from graphrag_n4j import GraphRAG
import json, re, random as _rnd
import retrieval as R
import csv as _csv
from collections import defaultdict

GRAPH_DIR = c.NOTEBOOKS_DATA / "graphrag"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
rprint("[bold]root :[/bold]", "../" + ROOT.name)
rprint("[bold]outputs :[/bold]", GRAPH_DIR.relative_to(ROOT))


root : ../energy-compliance-audit

---
## 1   Smoke test   one question end-to-end


In [2]:
rag = GraphRAG()
q1 = "Who must publish inside information under REMIT?"
t0 = time.time()
r1 = rag.query(q1)
dt = time.time() - t0

rprint("[bold green]GraphRAG.query()[/bold green]  %.1fs" % dt)
rprint("   answer length   :", len(r1.answer), "chars")
rprint("   sources         :", len(r1.sources), "  first:", r1.sources[:4])
rprint("   chunks          :", len(r1.chunks))
rprint("   entities        :", len(r1.entities), "  first:", r1.entities[:3])
rprint("   relationships   :", len(r1.relationships), "  first:", r1.relationships[:2])
rprint("   retrieval scores:", len(r1.retrieval_scores))
rprint("   cypher[:60]     :", r1.cypher[:60])
rprint()
rprint("[bold]answer:[/bold]")
print(r1.answer)


GraphRAG.query()  22.1s

answer length   : 1680 chars

sources         : 7   first:
['remit_1227_2011', 'remit_ii_2024_1106', 'data_act_2023_2854', 'entso_sogl_2017_1485']

chunks          : 60

entities        : 15   first:
[
    'acer_remit_guidance:term:consumption_capacity',
    'remit_1227_2011:term:consumption_capacity',
    'acer_remit_guidance:term:this'
]

relationships   : 57   first:
[
    'remit_1227_2011:article:1  CROSS_REFERENCES;DEFINED_IN@1hop',
    'remit_1227_2011:article:12  CROSS_REFERENCES;DEFINED_IN@1hop'
]

retrieval scores: 30

cypher[:60]     : MATCH (s), p=(s)--(n:Article|Preamble) WHERE s.linea

answer:

## Who Must Publish Inside Information Under REMIT

Under **Article 4** of Regulation (EU) No 1227/2011 (REMIT), **market participants** are the entities that **shall** publicly disclose inside information. The provision states:

> "Market participants shall publicly disclose in an effective and timely manner inside information which they possess in respect of bus…"

**Citation:** `remit_1227_2011:article:4`

### Supporting context from the graph

- The term **"inside information"** is an entity defined within REMIT (see `remit_1227_2011:article:2`, linked via the **DEFINED_IN** relationship at 1 hop).
- **Article 3** (Prohibition of insider trading) CROSS_REFERENCES the same concept, stating that "Persons who possess inside information in relation to a wholesale energy product shall be prohibited from" using or disclosing it improperly (`remit_1227_2011:article:3`).
- **Article 7** (Market monitoring) provides a limited exception: market participants retain "the right … to delay the d

---
## 2   Retrieval debugging (spec L624-648)

The spec asks to be able to see the full trace: **question -> seeds -> Cypher -> ranked results -> context -> LLM -> answer**. `result.debug()` exposes that; below we render it.

In [3]:
d = r1.debug()
rprint("[bold]debug dict keys:[/bold]", sorted(d.keys()))
rprint()
rprint("[bold]seeds:[/bold]")
for label, items in (d["seeds"] or {}).items():
    rprint("   %-12s %d seeds" % (label, len(items)))
    for s in items[:3]:
        rprint("      %-55s score=%.3f" % (s.get("lineage_id", "?")[:55], s.get("score", 0)))
rprint()
rprint("[bold]top-3 ranked:[/bold]")
for lid, hop, kinds in d["ranked"][:3]:
    rprint("   %-55s hop=%d kinds=%s" % (lid[:55], hop, kinds))
rprint()
rprint("[bold]cypher:[/bold]")
print(d["cypher"])
rprint()
rprint("[bold]prompt (first 600 chars):[/bold]")
print(d["prompt"][:600])


debug dict keys:
['answer', 'context', 'cypher', 'elapsed_ms', 'prompt', 'question', 'ranked', 'scores', 'seeds']

seeds:

Term         15 seeds

acer_remit_guidance:term:consumption_capacity           score=0.817

remit_1227_2011:term:consumption_capacity               score=0.817

...

remit_1227_2011:article:12                              hop=1 kinds=['CROSS_REFERENCES', 'DEFINED_IN']

cypher:

MATCH (s), p=(s)-[r*1..2]-(n:Article|Preamble) WHERE s.lineage_id IN $seed_ids   AND all(x IN r WHERE type(x) IN ["CROSS_REFERENCES", "AMENDS", "DEFINED_IN", "APPLIES_TO"])   AND n <> s WITH n,      min(length(p)) AS hop,      collect(DISTINCT s.lineage_id) AS seed_ids,      collect([x IN r | type(x)]) AS per_path RETURN n.lineage_id AS tid, hop, seed_ids,        reduce(acc=[], p IN per_path | acc + p) AS kinds


prompt (first 600 chars):

You are a legal-research assistant over a local knowledge
graph of European energy-market legislation and agency guidance.

Answer ONLY from the context supplied below.

Rules:
1. If the context is insufficient to answer, say "The supplied context is
   insufficient to answer this question." and stop.  Do not guess.
2. Do not invent articles, relationships, entities, obligations, or facts
   that are not in the context.
3. Cite the lineage_id (e.g. `remit_1227_2011:article:4`) of every
   important claim you make.
4. Where a graph relationship path supports the answer, name the
   relationship


---
## 3   Gold-set evaluation (retrieval quality, measured separately from answer quality)

Build a small deterministic gold set   article-titled nodes -> target chunk with the article title, plus DEFINED_IN edges -> chunks containing the term. Then measure **recall@k** (target lineage id appears in `result.chunks` within the top-k) and **safety** (the LLM never invents a lineage id not in the provenance section). Spec L591-606.

In [4]:

chunks_by_lid = {}
chunk_doc = {}
chunk_retriever = R.default_retriever()
for ch in chunk_retriever.chunks:
    chunks_by_lid[ch.lineage_id] = ch
    chunk_doc[ch.lineage_id] = ch.doc_id

# graph nodes
nodes = list(chunk_retriever.corpus["graph"].nodes.values())
articles = [n for n in nodes
              if n.get("kind") == "article" and n.get("title")
              and not re.match(r"^Article\s+\d+$", n["title"].strip())]

gold = []
for n in articles:
    lid = n["lineage_id"]
    if lid not in chunks_by_lid:
        continue
    doc, num = lid.split(":article:")
    title = n["title"]
    if len(title) < 12:
        continue
    gold.append((doc, f"{title}   who has this obligation under {doc}?", lid))

defined_terms = 0
edges_path = c.NOTEBOOKS_DATA / "graph" / "edges.jsonl"
for line in edges_path.read_text().splitlines():
    if not line.strip():
        continue
    e = json.loads(line)
    if e["kind"] != "DEFINED_IN" or e.get("unresolved"):
        continue
    doc = e["doc_id"]
    term = e.get("term")
    if not term or len(term) < 4:
        continue
    target = next(
        (x.lineage_id for x in chunk_retriever.chunks
         if x.doc_id == doc and term in x.text and len(x.text) < 900),
        None,
    )
    if target is None:
        continue
    gold.append((doc, f"What does '{term}' mean in {doc}?", target))
    defined_terms += 1
    if defined_terms >= 20:
        break

_rnd.seed(7)
gold = list(dict.fromkeys(gold))
_rnd.shuffle(gold)
gold = gold[:30]
rprint("[bold green]gold set:[/bold green]", len(gold), "(query, target) pairs")


gold set: 30 (query, target) pairs

In [5]:

K = 10          # target must appear in top-K chunks
SAFE_K = 5      # for safety eval (no-invention), top-5 chunk lids
rows = []
log = open(GRAPH_DIR / "graphrag_logs.jsonl", "w")
rec_at = defaultdict(int)
hit_at_5 = 0
no_invention_fail = 0
insufficient_count = 0
t_start = time.time()

for i, (doc, query, target) in enumerate(gold, 1):
    r = rag.query(query)
    top_k = [c["lineage_id"] for c in r.chunks][:K]
    top_5 = [c["lineage_id"] for c in r.chunks][:SAFE_K]
    hit5 = target in top_5
    hit10 = target in top_k[:10] if len(top_k) >= 10 else False
    if hit5:
        hit_at_5 += 1
    if hit5:
        rec_at[5] += 1
    if hit10:
        rec_at[10] += 1
    insufficient = r.answer.startswith("[INSUFFICIENT CONTEXT]")
    if insufficient:
        insufficient_count += 1
    # safety: every lineage id the LLM cited must be in provenance
    prov_set = set(lid for lid, _h, _k in r.debug()["ranked"])
    cited = re.findall(r"\w+:(?:article|term|entity|preamble|sentence):[\w\-]+", r.answer)
    invented = [c for c in cited if c not in prov_set]
    if invented:
        no_invention_fail += 1
    rows.append({
        "query": query, "doc": doc, "target": target,
        "hit5": hit5, "hit10": hit10,
        "insufficient": insufficient,
        "cited": len(cited), "invented": invented[:3] or [],
        "elapsed_ms": r.elapsed_ms,
    })
    log.write(json.dumps({"query": query, "target": target, "hit5": hit5,
                          "hit10": hit10, "invented": invented[:3] or [],
                          "elapsed_ms": r.elapsed_ms}) + "\n")
    rprint("  %2d/%d  %-70s [%s hit@5, %dms]" % (i, len(gold), query[:70],
           "OK" if hit5 else "miss", r.elapsed_ms))
log.close()
n = len(gold)
rprint("\n[bold]recall@5 :[/bold]", round(hit_at_5/n, 3))
rprint("[bold]recall@10:[/bold]", round(rec_at[10]/n, 3))
rprint("[bold]safety :[/bold]", n - no_invention_fail, "/", n, "no-invention")
rprint("[bold]insufficient-context (LLM declined to answer):[/bold]", insufficient_count)
rprint("[bold]total wall time:[/bold]", round(time.time() - t_start, 1), "s")


1/30  Establishment and mission of regional coordination centres   who has t

2/30  Conformity assessment bodies of third countries   who has this obligat

3/30  General provisions   who has this obligation under entso_ncrfg_2016_63

4/30  Implementation of the reference model   who has this obligation under  [OK hit@5, 6507ms]

5/30  Cross-zonal capacity calculation   who has this obligation under entso

...

recall@5 : 0.3

recall@10: 0.3

safety : 30 / 30 no-invention

insufficient-context (LLM declined to answer): 0

total wall time: 357.4 s

In [6]:
out = GRAPH_DIR / "graphrag_summary.csv"
with open(out, "w", newline="") as f:
    w = _csv.DictWriter(f, fieldnames=["query", "doc", "target", "hit5", "hit10",
                                      "insufficient", "cited", "invented", "elapsed_ms"])
    w.writeheader()
    w.writerows(rows)
rprint("[bold]artifact[/bold] :", out.relative_to(ROOT))

rprint("\n[bold]hand-off:[/bold]")
rprint("   entry point : from graphrag_n4j import GraphRAG")
rprint("   context     : src/graphrag_n4j/context.py")
rprint("   prompts     : SYSTEM_PROMPT, QUESTION_TEMPLATE")
rprint("   debug       : result.debug() -> {seeds, ranked, cypher, prompt, scores, ...}")


artifact : notebooks/data/graphrag/graphrag_summary.csv

hand-off:

entry point : from graphrag_n4j import GraphRAG

context     : src/graphrag_n4j/context.py

prompts     : SYSTEM_PROMPT, QUESTION_TEMPLATE

debug       : result.debug() -> {seeds, ranked, cypher, prompt, scores, ...}